Imports


In [8]:
from collections import deque
import re
import requests
from urllib.parse import urljoin, urlparse
from bs4 import BeautifulSoup
import time
import pickle
from pathlib import Path
from sentence_transformers import SentenceTransformer
import numpy as np

crawler


In [9]:
def crawl_products(start_points):
    visited = set()
    seen = set(start_points.keys())
    
    queue = deque([(url, 0, max_depth) for url, max_depth in start_points.items()])
    
    product_handles = set()
    base_domain = urlparse(next(iter(start_points))).netloc

    while queue:
        url, depth, max_depth = queue.popleft()
        
        print(f"[DEPTH {depth}/{max_depth}] A visitar: {url}")

        if depth > max_depth:
            continue
        
        visited.add(url)

        try:
            response = requests.get(url, timeout=5)
            soup = BeautifulSoup(response.text, 'html.parser')

            for link in soup.find_all('a', href=True):
                full_link = urljoin(url, link['href'])
                parsed = urlparse(full_link)

                
                if parsed.netloc != base_domain:
                    continue

                
                if any(x in full_link for x in [
                    "cart", "login", "wishlist", "translate",
                    "account", "checkout", "search"
                ]):
                    continue

                path = parsed.path

               
                if path.startswith("/product/"):
                    handle = path.split("/product/")[1]
                    handle = handle.split("?")[0]  # limpar params
                    handle = handle.strip("/")

                    if handle not in product_handles:
                        product_handles.add(handle)
                        print(f"  Produto encontrado: {handle}")

                    continue 

                
                if full_link not in seen:
                    seen.add(full_link)
                    queue.append((full_link, depth + 1, max_depth))

            print(f"  -> Total produtos: {len(product_handles)}")
            print(f"  -> Links vistos: {len(seen)}")

            time.sleep(1)

        except Exception as e:
            print(f"[ERRO] {url}: {e}")

    return product_handles


variaveis

In [10]:
start_points = {
        "https://www.inocos.com/": 1,
        "https://www.inocos.com/category": 2,
    }

products = crawl_products(start_points)


[DEPTH 0/1] A visitar: https://www.inocos.com/
  Produto encontrado: workshop-inocos-abrantes-mix-cosmeticos-or-11-maio
  Produto encontrado: workshop-inocos-pc-cosmetics-or-04-maio
  Produto encontrado: mystery-box-inocos
  Produto encontrado: box-verniz-gel-cateye-celestial-inocos
  Produto encontrado: iman-magnetico-inocos-cateye-5-em-1
  Produto encontrado: top-coat-inocos-cafe-natural
  Produto encontrado: multi-portable-nail-drill-inocos
  Produto encontrado: almofada-inocos-manicure-apoio-de-bracos
  Produto encontrado: box-glitter-biodegradavel-bioglitter-inocos
  Produto encontrado: glow-pro-nail-desk-lamp-candeeiro-inocos
  Produto encontrado: kit-complementos-manicure-russa-combinada-inocos
  -> Total produtos: 11
  -> Links vistos: 153
[DEPTH 0/2] A visitar: https://www.inocos.com/category
  -> Total produtos: 11
  -> Links vistos: 153
[DEPTH 1/1] A visitar: https://www.inocos.com/page/pontos-de-venda
  -> Total produtos: 11
  -> Links vistos: 184
[DEPTH 1/1] A visitar: htt

In [11]:
print("\n Produtos encontrados:")
for p in products:
    print(p)

print(f"\nTotal: {len(products)} produtos")


 Produtos encontrados:
verniz-gel-inocos-cateye-estrela-cadente
polyacrygel-inocos-6-nude-avela
lampada-inocos-open-flex-led-uv-10w
box-verniz-gel-naked-nails-inocos
verniz-gel-inocos-bosque-encantado
verniz-gel-inocos-viagem-de-comboio
verniz-gel-inocos-arrisca
like-gel-inocos-231-vermelho-papoila
fiber-base-gel-inocos-nude-perola
cureta-dupla-nailcare-pedicure-inocos-nc9
empurra-cuticulas-manicure-ponta-estreita-curva-maxi-inocos-nc1
verniz-gel-inocos-copo-d-agua
regulador-de-viscosidade-inocos
potal-flake-inocos-tamara-leitoso-com-flocos-ouro
glitter-biodegradavel-bioglitter-inocos-08-lilas-04-mm
paint-gel-inocos-amarelo
cover-up-inocos-suspiro-30g
glitter-xi-coracao-inocos-prata
like-gel-inocos-202-verde-matcha
removedor-de-verniz-inocos-150ml
oleo-de-cuticulas-inocos-de-framboesa-15ml
perfect-powder-inocos-amarelo-abelha-rainha-p58
paint-gel-inocos-vermelho
glitter-fantasia-inocos-mix-laranja-rosa-e-branco
verniz-gel-inocos-paprika-natura-lovers-spice-edition
builder-in-a-bottle-

Pré-processamento dos dados


In [12]:
def clean_handle(handle):
    remove_words = {"inocos", "hifans"}
    parts = handle.lower().split("-")
    cleaned = []
    for p in parts:
        if p not in remove_words:
            if re.match(r"^\d+[a-zA-Z]", p):
                p = re.sub(r"^(\d+).*", r"\1", p)
            cleaned.append(p)
    return " ".join(cleaned)

variaveis


In [13]:
dados_processados = [clean_handle(p) for p in products]
print(dados_processados)

['verniz gel cateye estrela cadente', 'polyacrygel 6 nude avela', 'lampada open flex led uv 10', 'box verniz gel naked nails', 'verniz gel bosque encantado', 'verniz gel viagem de comboio', 'verniz gel arrisca', 'like gel 231 vermelho papoila', 'fiber base gel nude perola', 'cureta dupla nailcare pedicure nc9', 'empurra cuticulas manicure ponta estreita curva maxi nc1', 'verniz gel copo d agua', 'regulador de viscosidade', 'potal flake tamara leitoso com flocos ouro', 'glitter biodegradavel bioglitter 08 lilas 04 mm', 'paint gel amarelo', 'cover up suspiro 30', 'glitter xi coracao prata', 'like gel 202 verde matcha', 'removedor de verniz 150', 'oleo de cuticulas de framboesa 15', 'perfect powder amarelo abelha rainha p58', 'paint gel vermelho', 'glitter fantasia mix laranja rosa e branco', 'verniz gel paprika natura lovers spice edition', 'builder in a bottle cover cafe', 'verniz gel verde glitter funk', 'fiber base gel rosa pessego leitoso', 'top coat no blue', 'almofada mini apoio de

## Criar embeddings e guardar tabela dos dados originais


In [14]:
DATA_DIR = Path("/Users/tomassilveira/Desktop/projeto/whatapp_to_excel/data")
DATA_DIR.mkdir(exist_ok=True)

model = SentenceTransformer("all-MiniLM-L6-v2", local_files_only=True)

with open(DATA_DIR / "prod.pkl", "wb") as f:
    pickle.dump(dados_processados, f)

produtos_norm = [p.lower() for p in dados_processados]

embeddings = model.encode(produtos_norm, convert_to_numpy=True)
np.save(DATA_DIR / "emb_prod.npy", embeddings)

print("Embeddings criados!")
print("Produtos guardados!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7284.23it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings criados!
Produtos guardados!


## guardar_tabela

In [15]:
DATA_DIR = Path.cwd().parent/"data"
DATA_DIR.mkdir(exist_ok=True)

with open(DATA_DIR / "prod.pkl", "wb") as f:
    pickle.dump(dados_processados, f)

print("Produtos guardados!")

Produtos guardados!
